# 04 - Previdenza complementare e confronti

Notebook esplorativo su fondi pensione, previdenza complementare e confronto internazionale.

Serve a misurare adesioni, asset, asset su PIL, cuneo fiscale e tassi di sostituzione quando le tabelle finali saranno popolate.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CODE_DIR = REPO_ROOT / 'code'
DATA_FINAL_DIR = REPO_ROOT / 'data' / 'final'
METADATA_DIR = REPO_ROOT / 'metadata'

if str(CODE_DIR) not in sys.path:
    sys.path.append(str(CODE_DIR))

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

## Parametri modificabili

In [ ]:
INDICATORE_ADERENTI = 'aderenti_previdenza_complementare'
INDICATORE_ASSET = 'asset_fondi_pensione'
INDICATORE_ASSET_PIL = 'asset_fondi_pensione_pil'
INDICATORE_CUNEO = 'cuneo_fiscale_lavoro'
INDICATORE_TASSO_SOSTITUZIONE = 'tasso_sostituzione'

In [ ]:
def read_csv_optional(path: Path) -> pd.DataFrame:
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()
    return pd.read_csv(path)


def filter_indicator(df: pd.DataFrame, indicatore_id: str) -> pd.DataFrame:
    if df.empty or 'indicatore_id' not in df.columns:
        return pd.DataFrame()
    return df[df['indicatore_id'].astype(str).eq(indicatore_id)].copy()


def plot_latest_by_country(df: pd.DataFrame, title: str, ylabel: str) -> None:
    if df.empty:
        print('Dati non disponibili:', title)
        return
    required = {'anno', 'paese', 'valore'}
    if not required.issubset(df.columns):
        print('Colonne richieste non disponibili:', required - set(df.columns))
        return
    data = df.copy()
    data['anno'] = pd.to_numeric(data['anno'], errors='coerce')
    data['valore'] = pd.to_numeric(data['valore'], errors='coerce')
    data = data.dropna(subset=['anno', 'paese', 'valore'])
    if data.empty:
        print('Nessun valore numerico:', title)
        return
    latest = data.sort_values('anno').groupby('paese', as_index=False).tail(1)
    latest = latest.sort_values('valore', ascending=False).head(20)
    plt.figure(figsize=(10, 6))
    plt.bar(latest['paese'].astype(str), latest['valore'])
    plt.title(title)
    plt.ylabel(ylabel)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## Dataset attesi

Questa cella mostra i dataset logici collegati a previdenza complementare e confronto internazionale.

In [ ]:
dataset_attesi = read_csv_optional(METADATA_DIR / 'dataset_attesi.csv')

if dataset_attesi.empty:
    print('dataset_attesi.csv non disponibile.')
else:
    mask = dataset_attesi['ambito'].astype(str).str.contains('previdenza|confronto|lavoro', case=False, na=False)
    display(dataset_attesi[mask])

## Caricamento tabelle finali

In [ ]:
previdenza = read_csv_optional(DATA_FINAL_DIR / 'tabella_previdenza_complementare.csv')
europa = read_csv_optional(DATA_FINAL_DIR / 'tabella_confronto_europeo.csv')
parametri = read_csv_optional(DATA_FINAL_DIR / 'tabella_parametri_sistema.csv')

display(pd.DataFrame({
    'tabella': ['previdenza_complementare', 'confronto_europeo', 'parametri_sistema'],
    'righe': [len(previdenza), len(europa), len(parametri)],
}))

## Previdenza complementare in Italia

Quando la tabella sara' popolata, qui si potranno leggere aderenti e asset.

In [ ]:
aderenti = filter_indicator(previdenza, INDICATORE_ADERENTI)
asset = filter_indicator(previdenza, INDICATORE_ASSET)

display(aderenti.head(20))
display(asset.head(20))

## Asset dei fondi pensione in rapporto al PIL

Indicatore utile per confrontare l Italia con paesi dove i fondi pensione hanno dimensioni molto maggiori.

In [ ]:
asset_pil = pd.concat([
    filter_indicator(previdenza, INDICATORE_ASSET_PIL),
    filter_indicator(europa, INDICATORE_ASSET_PIL),
], ignore_index=True)

display(asset_pil.head(30))
plot_latest_by_country(asset_pil, 'Asset dei fondi pensione in rapporto al PIL', 'Percentuale del PIL')

## Cuneo fiscale e tassi di sostituzione

Questi indicatori collegano pensioni, costo del lavoro e sostenibilita'.

In [ ]:
cuneo = filter_indicator(europa, INDICATORE_CUNEO)
tasso_sostituzione = pd.concat([
    filter_indicator(europa, INDICATORE_TASSO_SOSTITUZIONE),
    filter_indicator(parametri, INDICATORE_TASSO_SOSTITUZIONE),
], ignore_index=True)

display(cuneo.head(30))
display(tasso_sostituzione.head(30))

plot_latest_by_country(cuneo, 'Cuneo fiscale sul lavoro', 'Percentuale')
plot_latest_by_country(tasso_sostituzione, 'Tasso di sostituzione', 'Percentuale')

## Nota metodologica

Separare sempre COVIP, OECD ed Eurostat. Le tre fonti misurano cose diverse e servono per domande diverse. Prima di usare i risultati, controllare unita', paese, anno e definizione.